In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize, differential_evolution
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_8"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
#def format_inputdata(input_data):
#    res_formatted = [f"{r:.6f}" for r in input_data]
#    result = "-".join(res_formatted)
#    return result

In [3]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)
#print(data_in)
#print(data_out)

Week-01

In [4]:
X_init = data_in
y_init = data_out

bounds = np.array([(0.0, 1.0)] * X_init.shape[1])  # assume normalized [0,1] per dim
dim = X_init.shape[1]
y_best = y_init.max()

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)
# alpha = noise variance. If outputs are noisy, increase alpha (e.g. 1e-4 or tuned)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=5, random_state=0)
gp.fit(X_init, y_init)


# -------------------------
# Expected Improvement (EI)
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei  # note: maximize this

# Negative EI for minimizers
def neg_ei(x, gp, y_best, xi=0.01):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Acquisition optimization: many random starts + DE fallback
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    dim = bounds.shape[0]
    best_x = None
    best_val = 1e20

    # Random-start L-BFGS-B
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val = res.fun
            best_x = res.x

    # Try differential evolution as a global check (optional, slower)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best),
                                       bounds=bounds.tolist(),
                                       maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val = de_res.fun
            best_x = de_res.x
    except Exception:
        pass

    # clamp to bounds and return
    best_x = np.clip(best_x, bounds[:,0], bounds[:,1])
    return best_x

# -------------------------
# If you need a small batch (greedy sequential EI)
# -------------------------
def propose_batch(gp, y_best, bounds, batch_size=3):
    batch = []
    gp_copy = gp
    X_aug = X_init.copy()
    y_aug = y_init.copy()
    for i in range(batch_size):
        x_next = propose_location(gp_copy, y_aug.max(), bounds)
        # "Fake" evaluation: use GP predicted mean (Kriging believer)
        y_fake = gp_copy.predict(x_next.reshape(1, -1))[0]
        # Append to augmented set and refit GP (greedy sequential)
        X_aug = np.vstack([X_aug, x_next])
        y_aug = np.hstack([y_aug, y_fake])
        gp_copy = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
        gp_copy.fit(X_aug, y_aug)
        batch.append(x_next)
    return np.array(batch)

# -------------------------
# Run proposal
# -------------------------
x_next = propose_location(gp, y_best, np.array(bounds))
print("Suggested next point (continuous):", x_next)

# If some dims are categorical/encoded, round or map them here, e.g.:
# x_next[cat_index] = int(np.round(x_next[cat_index] * (num_categories-1)))
#
# Example: If dim 6 encodes optimizer 0..3:
# x_next[6] = int(np.round(x_next[6] * 3))

# -------------------------
# (Optional) propose a small batch
# -------------------------
batch = propose_batch(gp, y_best, np.array(bounds), batch_size=3)
print("Suggested batch of 3 points (continuous):\n", batch)


Suggested next point (continuous): [0.20349586 0.1680043  0.19510237 0.03586892 1.         1.
 0.22437016 0.49871938]
Suggested batch of 3 points (continuous):
 [[0.20349909 0.16800277 0.19510553 0.03587734 1.         1.
  0.22436863 0.4980955 ]
 [0.         0.25827122 0.01177767 0.02437755 1.         1.
  0.         1.        ]
 [0.17268136 0.         0.29077173 0.         1.         0.
  0.01284944 1.        ]]


In [5]:

# --- Initial data ---
#X_init = np.array([...])  # 30x6 array
#y_init = np.array([...])  # 30 outputs
X_init = data_in
y_init = data_out

y_best = y_init.max()  # best performance so far

# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization in scipy

# --- Optimize acquisition function ---
bounds = [(0,1)]*8
best_x = None
best_ei = float('inf')

for _ in range(20):  # multiple random starts
    x0 = np.random.rand(8)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)


Next hyperparameter set to try: [0.08878205 0.2881921  0.04408699 0.25667934 0.63480336 0.45896747
 0.21148188 0.52352432]


Week-02

In [6]:
#datain = X_init
#dataout = y_init
# -------------------------
# Existing data (your previous iteration)
# -------------------------
X_init = data_in
y_init = data_out

# Latest evaluated point (the one that gave 9.68029...)
X_new = np.array([0.141081,0.102602,0.176130,0.000000,1.000000,0.409759,0.147143,0.000000])
y_new = np.array([9.895763038795])

# -------------------------
# Step 1: Append new data
# -------------------------
X_all = np.vstack([X_init, X_new])
y_all = np.concatenate([y_init, y_new])

# -------------------------
# Step 2: Fit updated GP surrogate
# -------------------------
bounds = np.array([(0.0, 1.0)] * X_all.shape[1])
dim = X_all.shape[1]

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)

# Lower alpha for smooth deterministic behavior
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                              normalize_y=True, n_restarts_optimizer=5,
                              random_state=0)
gp.fit(X_all, y_all)

# -------------------------
# Step 3: Expected Improvement (EI) — exploitative mode
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.001):  # smaller xi = exploit
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

def neg_ei(x, gp, y_best, xi=0.001):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Step 4: Optimize acquisition to find next candidate
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    best_x, best_val = None, 1e20
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best, 0.001),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val, best_x = res.fun, res.x

    # Optional global search (safety)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best, 0.001),
                                        bounds=bounds.tolist(),
                                        maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val, best_x = de_res.fun, de_res.x
    except Exception:
        pass

    return np.clip(best_x, bounds[:,0], bounds[:,1])

# Compute updated best observed value
y_best = y_all.max()

# Get next suggested point
x_next = propose_location(gp, y_best, bounds)

print("Next suggested point (exploit mode):", x_next)

Next suggested point (exploit mode): [0.2332417  0.21025882 0.18553422 0.11013684 1.         1.
 0.28427944 1.        ]


Week-03

In [7]:
X_all = np.vstack([
    data_in, 
    [0.141081,0.102602,0.176130,0.000000,1.000000,0.409759,0.147143,0.000000],
    [0.233245,0.210250,0.185536,0.110135,1.000000,1.000000,0.284280,1.000000]
])

y_all = np.hstack([
    data_out, 
    9.895763038795,
    9.649813280537
])


bounds = np.array([(0.0, 1.0)] * X_all.shape[1])
dim = X_all.shape[1]

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)

# Lower alpha for smooth deterministic behavior
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                              normalize_y=True, n_restarts_optimizer=5,
                              random_state=0)
gp.fit(X_all, y_all)

# -------------------------
# Step 3: Expected Improvement (EI) — exploitative mode
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.001):  # smaller xi = exploit
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

def neg_ei(x, gp, y_best, xi=0.001):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Step 4: Optimize acquisition to find next candidate
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    best_x, best_val = None, 1e20
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best, 0.001),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val, best_x = res.fun, res.x

    # Optional global search (safety)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best, 0.001),
                                        bounds=bounds.tolist(),
                                        maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val, best_x = de_res.fun, de_res.x
    except Exception:
        pass

    return np.clip(best_x, bounds[:,0], bounds[:,1])

# Compute updated best observed value
y_best = y_all.max()

# Get next suggested point
x_next = propose_location(gp, y_best, bounds)

print(cf.format_inputdata(x_next))

0.213218-0.126016-0.227497-0.000000-1.000000-0.330615-0.288957-1.000000
